# Shoply Marketing & Customer Analytics
## Business Insights Using the Gold Layer

This notebook analyzes Shoply’s marketing performance and user behavior using the Gold layer of the data pipeline. The Gold layer contains curated, analytics-ready tables that combine cleaned behavioral data, campaign metadata, and customer attributes.

In [0]:
-- Business Question:
-- Which traffic sources bring in the most valuable traffic based on
-- revenue per session and conversion rate?

SELECT 
    utm_source,
    utm_medium,
    SUM(sessions) AS total_sessions,
    SUM(purchase_events) AS total_purchases,
    ROUND(SUM(purchase_revenue), 2) AS total_revenue,
    ROUND(SUM(purchase_revenue) / NULLIF(SUM(sessions), 0), 2) AS revenue_per_session,
    ROUND(SUM(purchase_events) / NULLIF(SUM(sessions), 0), 4) AS conversion_rate
FROM shoply_analytics.gold.dashboard_metrics
GROUP BY 
    utm_source,
    utm_medium
ORDER BY 
    revenue_per_session DESC,
    total_revenue DESC;


Facebook referral and organic traffic generated the highest revenue per session and nearly perfect conversion rates, indicating these users arrive with strong purchase intent. Email-based channels also showed high conversion performance, suggesting they reach engaged or returning customers. In contrast, search and organic traffic sources produced larger session volumes but lower revenue efficiency per user. This indicates that while search channels drive awareness and traffic, social referrals and email campaigns deliver the highest-value visitors.

In [0]:
-- Business Question:
-- Which campaigns generate the most traffic, purchases, and revenue?

SELECT
    campaign_name,
    SUM(sessions) AS total_sessions,
    SUM(purchase_events) AS total_purchases,
    ROUND(SUM(purchase_revenue), 2) AS total_revenue,
    ROUND(SUM(purchase_events) / NULLIF(SUM(sessions), 0), 4) AS conversion_rate
FROM shoply_analytics.gold.dashboard_metrics
GROUP BY
    campaign_name
ORDER BY
    total_revenue DESC,
    total_purchases DESC;

Campaign performance varies significantly across marketing efforts. The top campaigns generated the highest revenue and purchase activity while also maintaining strong conversion rates, indicating they successfully attracted high-intent users. Some campaigns produced large session volumes but comparatively fewer purchases, suggesting traffic quality differences across campaigns. Overall, the results highlight which campaigns are most effective at driving both traffic and revenue, helping guide future marketing investment toward the highest-performing initiatives.

In [0]:
-- Business Question:
-- How does user engagement and purchasing behavior differ across devices?

SELECT
    device_type,
    SUM(sessions) AS total_sessions,
    SUM(page_views) AS total_page_views,
    SUM(purchase_events) AS total_purchases,
    ROUND(SUM(page_views) / NULLIF(SUM(sessions), 0), 2) AS avg_pages_per_session,
    ROUND(SUM(purchase_events) / NULLIF(SUM(sessions), 0), 4) AS conversion_rate,
    ROUND(SUM(purchase_revenue), 2) AS total_revenue
FROM shoply_analytics.gold.dashboard_metrics
GROUP BY
    device_type
ORDER BY
    total_revenue DESC,
    total_sessions DESC;

Insights

Mobile devices generated the majority of traffic, sessions, and revenue, making them the primary driver of overall engagement and purchases. Desktop users showed the highest conversion rate, suggesting stronger purchase intent once users are browsing on larger screens. Tablet traffic is much smaller but still converts at a relatively strong rate, indicating a niche but engaged user group. Overall, the results suggest that while mobile drives volume, desktop users convert slightly more efficiently, highlighting the importance of optimizing both mobile usability and desktop checkout experiences.

In [0]:
-- Business Question:
-- Which product categories attract the most interest and drive the most revenue?

SELECT
    category_name,
    SUM(sessions) AS total_sessions,
    SUM(page_views) AS total_page_views,
    SUM(purchase_events) AS total_purchases,
    ROUND(SUM(purchase_revenue), 2) AS total_revenue,
    ROUND(SUM(purchase_events) / NULLIF(SUM(sessions), 0), 4) AS conversion_rate
FROM shoply_analytics.gold.dashboard_metrics
GROUP BY
    category_name
ORDER BY
    total_revenue DESC,
    total_sessions DESC;

The “unknown” category drives the majority of sessions, purchases, and revenue, suggesting that many sessions do not include a tracked category page visit or that category attribution is missing. Among the clearly identified categories, decor and kitchen generate the most revenue, indicating strong customer interest and purchasing behavior in these areas. Categories such as lighting and sale show the highest conversion rates, suggesting that users visiting these sections are highly purchase-oriented. Overall, the results suggest an opportunity to improve category tracking while prioritizing high-performing categories like decor, kitchen, and lighting in marketing campaigns and promotions.

In [0]:
-- Business Question:
-- Which campaigns deliver the strongest return relative to budget,
-- and where might budget need to be reallocated?

SELECT
    campaign_name,
    MAX(campaign_budget) AS campaign_budget,
    SUM(sessions) AS total_sessions,
    SUM(purchase_events) AS total_purchases,
    ROUND(SUM(purchase_revenue), 2) AS total_revenue,
    ROUND(SUM(purchase_revenue) / NULLIF(MAX(campaign_budget), 0), 2) AS revenue_per_budget_dollar,
    ROUND(SUM(purchase_events) / NULLIF(MAX(campaign_budget), 0), 4) AS purchases_per_budget_unit
FROM shoply_analytics.gold.dashboard_metrics
GROUP BY
    campaign_name
ORDER BY
    revenue_per_budget_dollar DESC,
    total_revenue DESC;

Some campaigns generate significantly higher revenue relative to their budget, indicating they are more efficient at converting marketing spend into revenue. These high-performing campaigns represent strong candidates for increased investment or scaling, as they demonstrate strong return on marketing spend. In contrast, campaigns with higher budgets but lower revenue efficiency may require optimization, targeting adjustments, or budget reallocation. Overall, analyzing revenue per budget dollar helps identify where marketing resources can be used most effectively.

In [0]:
-- Business Question:
-- Which customer segments contribute the most revenue and show
-- the strongest conversion behavior?

SELECT
    dc.age_group,
    dc.gender,
    dc.country,
    COUNT(DISTINCT f.customer_id) AS total_customers,
    SUM(f.sessions) AS total_sessions,
    SUM(f.purchase_events) AS total_purchases,
    ROUND(SUM(f.purchase_revenue), 2) AS total_revenue,
    ROUND(SUM(f.purchase_revenue) / NULLIF(COUNT(DISTINCT f.customer_id), 0), 2) AS revenue_per_customer,
    ROUND(SUM(f.purchase_events) / NULLIF(SUM(f.sessions), 0), 4) AS conversion_rate
FROM shoply_analytics.gold.fact_marketing_performance f
LEFT JOIN shoply_analytics.gold.dim_customer dc
    ON f.customer_id = dc.customer_id
GROUP BY
    dc.age_group,
    dc.gender,
    dc.country
ORDER BY
    total_revenue DESC,
    revenue_per_customer DESC;

Customer segments with the highest revenue tend to come from larger customer groups with strong purchasing activity, indicating that both segment size and engagement contribute to overall revenue. Segments showing higher revenue per customer suggest stronger individual purchasing power or repeat buying behavior. Some segments generate many sessions but comparatively fewer purchases, which may indicate browsing behavior rather than strong purchase intent. These findings can help prioritize high-value demographic groups for targeted marketing campaigns and personalized promotions.